# UK House Price Prediction AI Project

## Complete Jupyter Notebook Version

This notebook takes students through a complete data science and machine learning workflow:

1. Business understanding  
2. Loading the dataset  
3. Exploring the dataset  
4. Visualising the dataset  
5. Analysing missing values  
6. Handling missing data  
7. Feature engineering  
8. Encoding categorical features  
9. Training machine learning models  
10. Evaluating performance using RMSE  
11. Visualising model performance  
12. Generating predictions  
13. Saving the best model  
14. Creating a Streamlit deployment app  

The project uses a UK house price dataset and builds an AI model that predicts house prices.


## Lecturer Explanation

Say to students:

> Today we are going to behave like real data scientists. We will not jump straight into coding. We will follow a proper data science workflow.  
> First, we understand the business problem. Then we inspect the data, clean it, prepare it, train models, evaluate them, and finally deploy the model using Streamlit.  
> This is how real AI solutions are built in industry.


# 1. Install Required Libraries

Run this cell once if the required packages are not already installed.

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn joblib xgboost streamlit

# 2. Import Libraries

## Lecturer Explanation

Libraries are ready-made tools that help us do specific tasks.

- Pandas helps us work with tables.
- NumPy helps with numerical calculations.
- Matplotlib and Seaborn help us draw graphs.
- Scikit-learn gives us machine learning algorithms.
- XGBoost gives us an advanced boosting model.
- Joblib helps us save the trained model.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import joblib
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

# 3. Load the Dataset

## Lecturer Explanation

The dataset is the raw material for the machine learning project.

A machine learning model learns from examples inside a dataset. In this project, each row represents a house sale record, and the target we want to predict is the house price.

The code below first tries to load the dataset from the current Jupyter folder. If it is not found, it opens a file picker so you can select the CSV file from your computer.


In [ ]:
default_file = Path("UK_House_Price_Prediction_dataset_2015_to_2024.csv")

if default_file.exists():
    csv_path = str(default_file)
    print(f"Dataset found in current folder: {csv_path}")
else:
    print("Dataset not found in current folder.")
    print("A file picker will open. Please select your CSV dataset.")

    try:
        from tkinter import Tk
        from tkinter.filedialog import askopenfilename

        root = Tk()
        root.withdraw()
        root.attributes("-topmost", True)

        csv_path = askopenfilename(
            title="Select UK House Price CSV File",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )

        root.destroy()

        if not csv_path:
            raise FileNotFoundError("No file was selected.")

    except Exception as e:
        raise FileNotFoundError(
            "Could not open file picker. Please place the CSV file in the same folder as this notebook "
            "and name it UK_House_Price_Prediction_dataset_2015_to_2024.csv"
        ) from e

df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

# 4. Explore the Dataset

## Lecturer Explanation

Data exploration means looking at the dataset carefully before building any model.

We need to know the columns, the number of rows, the data types, the missing values, and the general structure of the data.


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nNumber of rows and columns:")
print(df.shape)

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

# 5. Understand Features and Target

## Lecturer Explanation

The target is what we want to predict.

In this project, the target is house price.

The features are the pieces of information we use to predict the price.

Simple explanation:

- Features = clues
- Target = answer
- Model = learns how clues connect to the answer


In [ ]:
target_column = "price"

if target_column not in df.columns:
    raise ValueError(f"The dataset must contain a '{target_column}' column.")

print("Target column:", target_column)
print("Example prices:")
print(df[target_column].head())

# 6. Analyse Missing Values

## Lecturer Explanation

Missing values are empty spaces in the dataset.

They can happen when information was not recorded properly. Before training the model, we must check missing values because many machine learning algorithms cannot handle blank values.


In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percent
})

missing_summary

# 7. Visualise the Dataset

## Lecturer Explanation

Visualisation helps us see patterns that may not be obvious in tables.

We will inspect the distribution of house prices and compare prices across property types.


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of UK House Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df["price"])
plt.title("House Price Outlier Check")
plt.xlabel("Price")
plt.show()

In [ ]:
if "property_type" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=df, x="property_type", y="price", estimator=np.mean)
    plt.title("Average House Price by Property Type")
    plt.xlabel("Property Type")
    plt.ylabel("Average Price")
    plt.show()
else:
    print("property_type column not found.")

# 8. Handle Missing Data

## Lecturer Explanation

There are different ways to handle missing data.

For this beginner project, we will fill missing text values with `Unknown` and drop rows where the target price is missing.


In [ ]:
clean_df = df.copy()

clean_df = clean_df.dropna(subset=["price"])

text_columns = clean_df.select_dtypes(include=["object"]).columns
for col in text_columns:
    clean_df[col] = clean_df[col].fillna("Unknown")

numeric_columns = clean_df.select_dtypes(include=["int64", "float64"]).columns
for col in numeric_columns:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

print("Missing values after cleaning:")
print(clean_df.isnull().sum())

# 9. Feature Engineering

## Lecturer Explanation

Feature engineering means creating useful new columns from existing data.

In this project, we extract year, month and quarter from the date column. We also extract postcode area from the postcode.


In [ ]:
if "date" in clean_df.columns:
    clean_df["date"] = pd.to_datetime(clean_df["date"], errors="coerce")
    clean_df["year"] = clean_df["date"].dt.year
    clean_df["month"] = clean_df["date"].dt.month
    clean_df["quarter"] = clean_df["date"].dt.quarter

if "postcode" in clean_df.columns:
    clean_df["postcode_area"] = clean_df["postcode"].astype(str).str.split().str[0]
else:
    clean_df["postcode_area"] = "Unknown"

for col in clean_df.select_dtypes(include=["object"]).columns:
    clean_df[col] = clean_df[col].fillna("Unknown")

for col in clean_df.select_dtypes(include=["int64", "float64"]).columns:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

clean_df.head()

# 10. Remove Columns Not Needed for Modelling

## Lecturer Explanation

Some columns are too detailed or not suitable for this beginner model. Full postcode and street can create too many unique values. We keep useful location information such as town, district, county and postcode area.


In [ ]:
columns_to_drop = []

for col in ["date", "postcode", "street", "locality"]:
    if col in clean_df.columns:
        columns_to_drop.append(col)

model_df = clean_df.drop(columns=columns_to_drop)

print("Dropped columns:", columns_to_drop)
print("Remaining columns:")
print(model_df.columns.tolist())

model_df.head()

# 11. Optional Outlier Handling

## Lecturer Explanation

Outliers are unusual values.

For house prices, some extremely high prices may represent luxury properties, errors, or rare cases. We remove the top 1% most expensive properties to make the model more stable for teaching.


In [ ]:
upper_limit = model_df["price"].quantile(0.99)
print("99th percentile price:", upper_limit)

model_df = model_df[model_df["price"] <= upper_limit]

print("Shape after removing extreme outliers:", model_df.shape)

# 12. Select Features and Target

## Lecturer Explanation

Now we separate our dataset into `X` and `y`.

- `X` contains the features.
- `y` contains the target price.


In [ ]:
X = model_df.drop(columns=["price"])
y = model_df["price"]

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget column: price")
print("X shape:", X.shape)
print("y shape:", y.shape)

# 13. Identify Categorical and Numerical Features

## Lecturer Explanation

Categorical columns are text columns such as town or property type. Numerical columns are number columns such as year, month and quarter.

We need to encode categorical columns before training the model.


In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

# 14. Encode Categorical Features

## Lecturer Explanation

Encoding means converting text into numbers.

One-hot encoding creates new columns for each category. This allows the machine learning model to use text-based information.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

print("Preprocessor created successfully.")

# 15. Split the Data into Training and Testing Sets

## Lecturer Explanation

We train the model on 80% of the data and test it on 20%.

This helps us check whether the model can make predictions on new unseen data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

# 16. Define Machine Learning Models

## Lecturer Explanation

We will train four machine learning models:

- Linear Regression
- Random Forest
- Gradient Boosting
- XGBoost


In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        objective="reg:squarederror"
    )
}

print("Models defined successfully.")
print(list(models.keys()))

# 17. Train and Evaluate the Models

## Lecturer Explanation

Training means the model learns patterns from the training data.

Evaluation means we compare predicted prices with real prices.

RMSE tells us how far the predictions are from the real values. Lower RMSE means better performance.


In [ ]:
results = {}
trained_models = {}

best_model = None
best_model_name = None
best_rmse = float("inf")

for model_name, model in models.items():
    print("=" * 60)
    print(f"Training model: {model_name}")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results[model_name] = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }

    trained_models[model_name] = pipeline

    print(f"RMSE: £{rmse:,.2f}")
    print(f"MAE: £{mae:,.2f}")
    print(f"R2 Score: {r2:.4f}")

    if rmse < best_rmse:
        best_rmse = rmse
        best_model = pipeline
        best_model_name = model_name

print("=" * 60)
print("Best model:", best_model_name)
print("Best RMSE:", f"£{best_rmse:,.2f}")

# 18. Create a Model Performance Table

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="RMSE")
results_df

# 19. Visualise Model Performance

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=results_df.index, y=results_df["RMSE"])
plt.title("Model Comparison using RMSE")
plt.xlabel("Machine Learning Model")
plt.ylabel("RMSE")
plt.xticks(rotation=30)
plt.show()

# 20. Generate Predictions with the Best Model

In [ ]:
example_property = X_test.iloc[[0]]

print("Example property details:")
display(example_property)

actual_price = y_test.iloc[0]
predicted_price = best_model.predict(example_property)[0]

print(f"Actual price: £{actual_price:,.2f}")
print(f"Predicted price: £{predicted_price:,.2f}")

# 21. Create a Custom Prediction Example

In [ ]:
custom_property = X.iloc[[0]].copy()

if "year" in custom_property.columns:
    custom_property["year"] = 2024

if "month" in custom_property.columns:
    custom_property["month"] = 6

if "quarter" in custom_property.columns:
    custom_property["quarter"] = 2

print("Custom property used for prediction:")
display(custom_property)

custom_prediction = best_model.predict(custom_property)[0]

print(f"Predicted house price: £{custom_prediction:,.2f}")

# 22. Save the Best Model and Feature Options

In [ ]:
joblib.dump(best_model, "best_house_price_model.pkl")

feature_options = {}

for col in X.columns:
    if col in categorical_features:
        feature_options[col] = sorted(X[col].astype(str).unique().tolist())
    else:
        feature_options[col] = {
            "min": float(X[col].min()),
            "max": float(X[col].max()),
            "median": float(X[col].median())
        }

joblib.dump(feature_options, "feature_options.pkl")

print("Best model saved as best_house_price_model.pkl")
print("Feature options saved as feature_options.pkl")

# 23. Create a Streamlit App from the Notebook

The next cell writes a complete `app.py` file. After running it, open a terminal in the same folder and run:

```bash
streamlit run app.py
```


In [ ]:
streamlit_code = """
import streamlit as st
import pandas as pd
import joblib

model = joblib.load("best_house_price_model.pkl")
feature_options = joblib.load("feature_options.pkl")

st.set_page_config(
    page_title="UK House Price Prediction",
    page_icon="🏠",
    layout="centered"
)

st.title("UK House Price Prediction System")

st.write(
    "This app predicts UK house prices using a trained machine learning model. "
    "Enter the property details below and click Predict."
)

st.warning(
    "This prediction is for educational purposes only and should not be treated "
    "as a professional property valuation."
)

st.header("Enter Property Details")

input_data = {}

for feature, options in feature_options.items():
    if isinstance(options, list):
        input_data[feature] = st.selectbox(feature, options)
    else:
        min_value = options["min"]
        max_value = options["max"]
        median_value = options["median"]

        if float(min_value).is_integer() and float(max_value).is_integer():
            input_data[feature] = st.number_input(
                feature,
                min_value=int(min_value),
                max_value=int(max_value),
                value=int(median_value),
                step=1
            )
        else:
            input_data[feature] = st.number_input(
                feature,
                min_value=float(min_value),
                max_value=float(max_value),
                value=float(median_value)
            )

input_df = pd.DataFrame([input_data])

st.subheader("Input Summary")
st.dataframe(input_df)

if st.button("Predict House Price"):
    prediction = model.predict(input_df)[0]
    st.success(f"Estimated House Price: £{prediction:,.2f}")

st.subheader("How It Works")

st.write(
    "The model was trained using historical UK house price data. "
    "It learned relationships between property information and sale prices."
)
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print("Streamlit app created as app.py")

# 24. Final Student Reflection Questions

Ask students:

1. What was the business problem?
2. What was the target variable?
3. What were the features?
4. Why did we encode categorical data?
5. Why did we split the data into training and testing sets?
6. Which model performed best?
7. What does RMSE mean?
8. Why is Streamlit useful for deployment?
9. What are the limitations of this model?
10. How could we improve the model in the future?


# 25. Closing Explanation

Say to students:

> Today you completed a full AI solution.  
> You moved from business understanding to deployment.  
> This is the same workflow used by real data scientists and machine learning engineers.  
> Machine learning is not only about training a model. It is about solving a real problem using data, careful preparation, evaluation and deployment.
